## 🛒 Automated Recommendation & Predictive Couponing Engine
This project implements a high-performance retail analytics pipeline using Polars. It combines Market Basket Analysis (MBA) with Customer Segmentation to trigger personalized promotions at the optimal time.

### 1. Data Preparation & Segmentation Joining
We start by merging raw transaction data with pre-calculated customer clusters. This allows the engine to generate recommendations tailored to specific shopping behaviors (e.g., Premium, Budget, or Eco-friendly shoppers).

Logic: Inner join on user_id.

Optimization: Using LazyFrames to minimize memory footprint.

### 2. Optimized Market Basket Analysis (MBA)
Instead of a generic analysis, we run an association rule algorithm per cluster.

⚙️ Key Optimizations:
Frequency Filtering: We ignore rare products (long tail) to focus on statistically significant patterns.

Breadth Limiting: We limit baskets to the first 20 items to prevent computational explosion.

Metrics: We calculate Support, Confidence, and Lift to measure the strength of product relationships.

### 3. Predictive Repurchase Timing
A promotion is only effective if sent when the customer is about to run out of a product.

Average Cycle: We calculate the mean days_since_prior_order for every product within each cluster.

The "80% Rule": We schedule the promotion to arrive at 80% of the typical cycle to capture the customer's intent just before they look elsewhere.

### 4. Final Campaign Orchestration
The final step generates a unique, high-value offer for each user by:

Scanning History: Identifying the last 5 products purchased.

Matching Rules: Finding the best "Target Product" based on the highest Opportunity Score.

Dynamic Discounting:

20% OFF: High-discovery products (Lift > 10).

10% OFF: Standard replenishment items.

In [1]:
import polars as pl

In [2]:
transactions = pl.scan_parquet("../data/processed/df_final_for_pipeline.parquet")
clusters_map = pl.scan_parquet("../data/processed/customer_segmentation_results.parquet").select(["user_id", "cluster"])
df_combined = transactions.join(clusters_map, on="user_id", how="inner")

print("✅ df_combined ready for MBA")

✅ df_combined ready for MBA


In [3]:
def get_association_rules_optimized(df_lazy, cluster_id, min_support_count=50):
    print(f"--- Computation for Cluster n°{cluster_id} ---")

    #1. We limit data: We ignore rare products AND we limit basket size.
    data = (
        df_lazy
        .filter(pl.col("cluster") == cluster_id)
        .select(["order_id", "product_name"])
        .with_columns(
            n = pl.lit(1).cum_sum().over("order_id")
        )
        .filter(pl.col("n") <= 20)
        .collect()
    )

    #2. Only keep products that are used frequently.
    counts = data["product_name"].value_counts()
    frequent_products = counts.filter(pl.col("count") >= min_support_count)["product_name"]

    filtered_data = data.filter(pl.col("product_name").is_in(frequent_products))
    paires = (
        filtered_data.join(filtered_data, on="order_id")
        .filter(pl.col("product_name") < pl.col("product_name_right"))
        .group_by(["product_name", "product_name_right"])
        .agg(pl.len().alias("pair_count"))
        .filter(pl.col("pair_count") >= min_support_count)
    )

    # 4. Metrics calculation
    total_orders = data["order_id"].n_unique()
    rules = (
        paires
        .join(counts, left_on="product_name", right_on="product_name")
        .rename({"count": "count_A"})
        .join(counts, left_on="product_name_right", right_on="product_name")
        .rename({"count": "count_B"})
        .with_columns([
            (pl.col("pair_count") / total_orders).alias("support"),
            (pl.col("pair_count") / pl.col("count_A")).alias("confidence"),
            ((pl.col("pair_count") / total_orders) /
             ((pl.col("count_A") / total_orders) * (pl.col("count_B") / total_orders))
            ).alias("lift")
        ])
    )
    return rules

all_rules_list = []

# --- LOOP ---
for c_id in [0, 1, 2]:
    cluster_rules = get_association_rules_optimized(df_combined, cluster_id=c_id, min_support_count=15)

    if cluster_rules.height > 0:
        cluster_rules = cluster_rules.with_columns(pl.lit(c_id).alias("cluster"))
        all_rules_list.append(cluster_rules)
    else:
        print(f"⚠️ No rules has been found for cluster {c_id} with that threshold.")

# --- EXPORT ---
if len(all_rules_list) > 0:
    all_rules = pl.concat(all_rules_list)
    top_recommendations = (
        all_rules
        .with_columns([
            (pl.col("lift") * pl.col("confidence")).alias("opportunity_score")
        ])
        .filter(pl.col("lift") > 1.2)
        .sort(["cluster", "opportunity_score"], descending=True)
        .group_by("cluster")
        .head(100)
    )

    # Final export
    output_path = "../data/outputs/next_product_promo_rules.parquet"
    top_recommendations.write_parquet(output_path)

    output_csv_path = "../data/outputs/next_product_promo_rules.csv"
    top_recommendations.write_csv(output_csv_path, separator=",")

    print(f"🚀 Success! Recommendation engine saved here: {output_path}")
    display(top_recommendations)
else:
    print("❌ Error: No rules could be generated for all clusters.")

--- Computation for Cluster n°0 ---


C:\Users\natha\AppData\Local\Temp\ipykernel_41564\3963130166.py:20: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  filtered_data = data.filter(pl.col("product_name").is_in(frequent_products))


--- Computation for Cluster n°1 ---
--- Computation for Cluster n°2 ---
🚀 Success! Recommendation engine saved here: ../data/outputs/next_product_promo_rules.parquet


cluster,product_name,product_name_right,pair_count,count_A,count_B,support,confidence,lift,opportunity_score
i32,str,str,u32,u32,u32,f64,f64,f64,f64
2,"""Moisturizing Facial Wash""","""Moisturizing Non-Drying Facial…",21,22,22,0.00003,0.954545,30404.615702,29022.587716
2,"""Dark Chocolate Raspberry Skinn…","""Skinny Dipped Dark Chocolate A…",15,18,21,0.000021,0.833333,27807.698413,23173.082011
2,"""Pate in Natural Juices Beef En…","""Premium Pate Wet Cat Food - Ch…",22,23,39,0.000031,0.956522,17186.831661,16439.578111
2,"""Banana Greek Nonfat Yogurt""","""Coconut Quinoa Yogurt""",16,21,25,0.000023,0.761905,21356.312381,16271.4761
2,"""Apple Cinnamon Yogurt""","""Banana Greek Nonfat Yogurt""",17,25,21,0.000024,0.68,22691.081905,15429.935695
…,…,…,…,…,…,…,…,…,…
0,"""Banana""","""Honeycrisp Apple""",16,239,45,0.012204,0.066946,1.950349,0.130567
0,"""Banana""","""Organic Whole Milk""",18,239,61,0.01373,0.075314,1.61863,0.121905
0,"""Banana""","""Large Lemon""",18,239,67,0.01373,0.075314,1.473678,0.110988


In [4]:
# 1. Loading transactions with dates
# Ensure you have “days_since_prior_order” or a date column
df = pl.scan_parquet("../data/processed/df_final_for_pipeline.parquet")
clusters = pl.scan_parquet("../data/processed/customer_segmentation_results.parquet").select(["user_id", "cluster"])

# 2. Calculation of the average repurchase cycle by product and by cluster
# The cumulative time frame is calculated for each product
purchase_cycles = (
    df.join(clusters, on="user_id")
    .group_by(["cluster", "product_name"])
    .agg([
        pl.col("days_since_prior_order").mean().alias("avg_days_between_purchases"),
        pl.len().alias("purchase_count")
    ])
    # We only keep products that are purchased often enough to be significant.
    .filter(pl.col("purchase_count") > 10)
    .collect()
)

#3. Definition of the promotion window (e.g. 80% of the average cycle)
promo_timing = purchase_cycles.with_columns(
    send_promo_after_days = (pl.col("avg_days_between_purchases") * 0.8).round(0)
)

promo_timing.write_csv("../data/outputs/product_timing_by_cluster.csv")
print("✅ Redemption cycles calculated!")

✅ Redemption cycles calculated!


### Discount generation

In [5]:
import polars as pl

df_transactions = pl.scan_parquet("../data/processed/df_final_for_pipeline.parquet")
df_clusters = pl.scan_parquet("../data/processed/customer_segmentation_results.parquet").select(["user_id", "cluster"])

rules = pl.read_csv("../data/outputs/next_product_promo_rules.csv")
trigger_products = rules["product_name"].unique().to_list()

# Transactions are filtered BEFORE sorting
# Only lines where the product can trigger a rule are kept
last_purchases = (
    df_transactions
    .filter(pl.col("product_name").is_in(trigger_products))
    .sort("order_number", descending=True)
    .group_by("user_id")
    .head(5)
    .join(df_clusters, on="user_id")
    .select(["user_id", "product_name", "cluster"])
    .collect()
)

campaign_plans = (
    last_purchases.join(rules, on=["product_name", "cluster"], how="inner")
    .with_columns([
        (pl.col("lift") * pl.col("confidence")).alias("score_final")
    ])
    .sort("score_final", descending=True)
    .group_by("user_id")
    .first() # We keep the best of the 5 options for the user.
)

timing = pl.read_csv("../data/outputs/product_timing_by_cluster.csv")
final_offers = (
    campaign_plans.join(
        timing.select(["cluster", "product_name", "send_promo_after_days"]),
        left_on=["cluster", "product_name_right"],
        right_on=["cluster", "product_name"],
        how="left"
    )
    .with_columns(
        discount_value = pl.when(pl.col("lift") > 10).then(pl.lit("20%")).otherwise(pl.lit("10%"))
    )
)

final_offers.write_csv("../data/outputs/final_campaign_offers.csv")
print(f"🚀 Done ! {final_offers.height} generated offers.")

🚀 Done ! 50649 generated offers.


**user_id**: Your customer's unique identifier. This is the key that allows you to know who to send the email or notification to.

**cluster**: The segment to which the customer belongs (0: Premium Healths, 1: Daily Economisers, 2: Budget-Healthy Mix). This defines the ‘tone’ of communication and the type of products they like.

**product_name**: This is the product that the customer has recently purchased (among their last 5 purchases). This is the lever. We use this purchase to justify the promotion.

**product_name_right**: This is the recommended product. It is the target of your promotion. The algorithm has determined that if the customer buys A, they are likely to like B.

**pair_count**: The number of times product A and product B were purchased together in the same basket within this cluster.

**count_A**: The total number of times product A was purchased in this cluster.

**count_B**: The total number of times product B was purchased in this cluster.

**support**: The popularity of the pair (A+B) across all orders in the cluster. The higher it is, the more frequent the association.

**confidence**: The probability. If the value is 0.4, it means that there is a 40% chance that a customer buying A will also buy B.

**lift**: The strength of the association. A lift of 5 means that the customer is 5 times more likely to buy B if they have bought A, compared to an average customer. This is the strongest indicator of relevance.

**opportunity_score**: Calculated during the MBA (lift * confidence). It is used to rank the best theoretical rules by cluster.

**score_final**: This is the score calculated in the final step. It is used to choose, from among the last five products purchased by the customer, which one generates the most powerful recommendation. This score was used to eliminate duplicates and keep only one offer per user_id.

**send_promo_after_days**: Timing. This is the number of days to wait after the last purchase before sending the promotion. It is based on the average repurchase cycle for product B to avoid offering a product that the customer still has in stock.

**discount_value**: Amount.

20%: For products with a very high lift (we want to encourage discovery).

10%: For more traditional combinations (we just want to encourage repurchase).